In [1]:
%matplotlib qt
import os
import cv2
import mne
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as patches
from scipy.signal import hilbert
from PyEMD import EMD

from neuronol_signalprocessing import *
# from neuronol_power import featureFunc_bandAbsolutePower, featureFunc_bandRelativePower, \
#                            featureFunc_centroidFrequencyBand, featureFunc_dominantFrequency

# plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = [14, 10]
plt.rcParams['font.size'] = 14
plt.rcParams['figure.max_open_warning'] = 100

In [2]:
def computeFourierPowerSpectrum1D(x, Fs, to_window=True, to_plot=True):
    '''
    Compute the Fourier power spectrum of the signal `x` sampled
    at frequency `Fs`.
    '''
    n = len(x)
    
    # Windowing
    if to_window:
        w = np.hamming(n)
        x = w * x
    
    # FFT
    xhat = np.fft.fft(x, n)
    psd = np.real(xhat * np.conj(xhat) / n) # power spectral density
    freq = Fs / n * np.arange(n)
    k = np.arange(np.floor(n / 2), dtype='int') # only plot the first half of frequencies
    
    # Plot power spectrum
    if to_plot:
        plt.figure()
        plt.plot(freq[k], psd[k], color='k', lw=2)
        plt.show()
        
    return psd[k], freq[k]

# Given

In [ ]:
dir_data = os.path.expanduser('~/data/wash-u/preprocessed/v2/')
dir_results = os.path.expanduser('~/research/results/wash-u/demos')

# Power spectrum demo

In [ ]:
fname_raw = '7101_4_rest1_ec_ica_ssp_eeg.fif'
fpath_raw = os.path.join(dir_data, fname_raw)

# Read preprocessed EEG data
raw = mne.io.read_raw_fif(fpath_raw)
Fs = np.floor(raw.info['sfreq'])

# Find intervals for eyes-closed resting state
ec_end = [raw.annotations.onset[i] for i, annt in
          enumerate(raw.annotations.description) if annt in ('1', '3')]
ec_intvl = {}
ec_intvl['EC1'] = np.floor([ec_end[0] - 63, ec_end[0] - 3])
ec_intvl['EC2'] = np.floor([ec_end[1] - 63, ec_end[1] - 3])

# Crop raw to include an epoch of 10 seconds from 'EC1' interval
tmin, tmax = ec_intvl['EC1']
cropped_raw = raw.copy().crop(tmin=tmin, tmax=tmax)
T_max = 10
raw_data = cropped_raw.get_data(picks='eeg')
raw_data = raw_data[:, :int(Fs * T_max)]
print('Shape of raw_data =', raw_data.shape)

# Take only the first channel data (i.e., 'O2') and make an array for time
x = raw_data[0, :] * 1e6 # in μV
t = np.arange(0, T_max, 1 / Fs)

# Calculate Fourier power spectrum
spec_fft, f_fft = computeFourierPowerSpectrum1D(x, Fs, to_plot=False)

# # Perform EMD-HHT
# imfs = perform_EMD(x, t=t, plot_emd=False)
# print('Found a total of %02d IMFs' % len(imfs))
# C = imfs[:-1]
# hht, t_hht, f_hht, f_hht_marginal, spec_hht_marginal = calculate_hilbert_spectrum(C, t, Fs,
#                                     compute_power_spec=True, smoothing_downsample_freq=False,
#                                     smoothing_gauss_filt=False, plot_inst_freq=False)

In [ ]:
# Make custom demo plot
fpath_fig = os.path.join(dir_results, 'Demo-EEGPowerSpec.pdf')

fig, ax = plt.subplots()

ax.plot(f_hht_marginal, spec_hht_marginal, 'k', lw=3)
ymax = np.max(spec_hht_marginal) * 1.1

ax.set_xlim([0, 30])
ax.set_ylim([0, ymax])

# Use colormap
colors = cm.viridis_r(np.linspace(0, 1, 5))

# Use theme-colors: 'mediumturquoise', 'magenta', 'steelblue', 'darkorchid', 'aliceblue'
# colors = ['deeppink', 'darkorchid', 'mediumturquoise', 'darkblue', 'none']

rect_del = patches.Rectangle((0, 0), 3, ymax, color=colors[0], alpha=0.8)
rect_the = patches.Rectangle((3, 0), 5, ymax, color=colors[1], alpha=0.8)
rect_alp = patches.Rectangle((8, 0), 5, ymax, color=colors[2], alpha=0.8)
rect_bet = patches.Rectangle((13, 0), 17, ymax, color=colors[3], alpha=0.8)

ax.add_patch(rect_del)
ax.add_patch(rect_the)
ax.add_patch(rect_alp)
ax.add_patch(rect_bet)

plt.axis('off')
plt.tight_layout()
plt.savefig(fpath_fig, bbox_inches='tight')
plt.savefig(fpath_fig[:-4] + '.png', bbox_inches='tight') # also save as png
plt.show()